# Data Drift Analysis: Credit Scoring Model

This notebook detects whether production data has drifted from training data using statistical tests and visualizations. Data drift occurs when the real-world data your model sees differs significantly from the data it was trained on — a silent failure that degrades model performance.

**What we'll do:**
1. Load reference (training) data
2. Generate realistic production data with drift
3. Visualize distributions (before/after)
4. Run statistical drift tests with Evidently AI
5. Interpret results and recommend actions

## Section 1: Load Reference Data

The reference data is the training data baseline. We compare production data against it to detect drift.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load reference data (training data baseline)
reference_data = pd.read_csv('data/reference_data.csv')
print(f"Reference data loaded: {reference_data.shape}")
print(f"Features: {list(reference_data.columns)}")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print("Reference data summary:")
print(reference_data.describe())

## Section 2: Simulate Production Data with Realistic Drift

Creating production data that looks like what we'd see "6 months later" with realistic shifts:
- EXT_SOURCE scores shift slightly (credit bureau algorithms updated)
- Financial amounts increase (inflation, economic growth)
- Applicant demographics shift (younger customers join platform)

This demonstrates how drift manifests in real-world scenarios.

np.random.seed(123)
n_prod = 5000

# Shift 1: External credit bureau scores (0.05 shift for EXT_SOURCE_1, 0.03 for EXT_SOURCE_2)
# This simulates recalibration of credit bureau scoring models
prod_ext_1 = reference_data['EXT_SOURCE_1'].dropna().sample(n_prod, replace=True).values + \
    np.random.normal(0.05, 0.02, n_prod)
prod_ext_1 = np.clip(prod_ext_1, 0, 1)

prod_ext_2 = reference_data['EXT_SOURCE_2'].dropna().sample(n_prod, replace=True).values + \
    np.random.normal(0.03, 0.01, n_prod)
prod_ext_2 = np.clip(prod_ext_2, 0, 1)

# EXT_SOURCE_3 stays stable (not all features drift simultaneously)
prod_ext_3 = reference_data['EXT_SOURCE_3'].dropna().sample(n_prod, replace=True).values
prod_ext_3 = np.clip(prod_ext_3, 0, 1)

print("EXT_SOURCE shifts:")
print(f"  EXT_SOURCE_1: {reference_data['EXT_SOURCE_1'].mean():.3f} → {prod_ext_1.mean():.3f}")
print(f"  EXT_SOURCE_2: {reference_data['EXT_SOURCE_2'].mean():.3f} → {prod_ext_2.mean():.3f}")
print(f"  EXT_SOURCE_3: {reference_data['EXT_SOURCE_3'].mean():.3f} → {prod_ext_3.mean():.3f} (stable)")

In [ ]:
# Shift 2: Financial features increase due to inflation
# Income +8%, Credit +12%, Annuity +10%, Goods +15%
prod_income = reference_data['AMT_INCOME_TOTAL'].sample(n_prod, replace=True).values * 1.08
prod_credit = reference_data['AMT_CREDIT'].sample(n_prod, replace=True).values * 1.12
prod_annuity = reference_data['AMT_ANNUITY'].sample(n_prod, replace=True).values * 1.10
prod_goods = reference_data['AMT_GOODS_PRICE'].sample(n_prod, replace=True).values * 1.15

print("\nFinancial features (inflation simulation):")
print(f"  AMT_INCOME_TOTAL: +8%")
print(f"  AMT_CREDIT: +12%")
print(f"  AMT_ANNUITY: +10%")
print(f"  AMT_GOODS_PRICE: +15%")

## Shift 3: Applicant Demographics

Marketing campaign targets younger customers. 5-year average age decrease indicates demographic shift.

In [ ]:
# Younger customer base (5-year average age decrease)
prod_age = reference_data['AGE_YEARS'].sample(n_prod, replace=True).values - \
    np.random.uniform(0, 5, n_prod)
prod_age = np.clip(prod_age, 20, 70)

print(f"\nApplicant age shifts (younger customers):")
print(f"  Before: {reference_data['AGE_YEARS'].mean():.1f} years")
print(f"  After: {prod_age.mean():.1f} years")
print(f"  Shift: -{reference_data['AGE_YEARS'].mean() - prod_age.mean():.1f} years")

## Assemble Production Dataset

Combine all features (including engineered ratios) into a complete DataFrame that matches the training data structure.

In [ ]:
current_data = pd.DataFrame({
    'EXT_SOURCE_1': prod_ext_1,
    'EXT_SOURCE_2': prod_ext_2,
    'EXT_SOURCE_3': prod_ext_3,
    'AMT_INCOME_TOTAL': prod_income,
    'AMT_CREDIT': prod_credit,
    'AMT_ANNUITY': prod_annuity,
    'AMT_GOODS_PRICE': prod_goods,
    'AGE_YEARS': prod_age,
    'CODE_GENDER': reference_data['CODE_GENDER'].sample(n_prod, replace=True).values,
    'FLAG_OWN_CAR': reference_data['FLAG_OWN_CAR'].sample(n_prod, replace=True).values,
    'FLAG_OWN_REALTY': reference_data['FLAG_OWN_REALTY'].sample(n_prod, replace=True).values,
    'CNT_CHILDREN': reference_data['CNT_CHILDREN'].sample(n_prod, replace=True).values,
    'YEARS_EMPLOYED': reference_data['YEARS_EMPLOYED'].sample(n_prod, replace=True).values,
    'YEARS_ID_PUBLISH': reference_data['YEARS_ID_PUBLISH'].sample(n_prod, replace=True).values,
    'EDUCATION_LEVEL': reference_data['EDUCATION_LEVEL'].sample(n_prod, replace=True).values,
})

# Compute engineered features (same as training)
current_data['CREDIT_INCOME_RATIO'] = current_data['AMT_CREDIT'] / (current_data['AMT_INCOME_TOTAL'] + 1)
current_data['ANNUITY_INCOME_RATIO'] = current_data['AMT_ANNUITY'] / (current_data['AMT_INCOME_TOTAL'] + 1)
current_data['CREDIT_GOODS_RATIO'] = current_data['AMT_CREDIT'] / (current_data['AMT_GOODS_PRICE'] + 1)

print(f"Production dataset created: {current_data.shape}")
print(f"All 18 features present: {current_data.shape[1] == 18}")

## Section 3: Visual Distribution Comparison

Side-by-side histograms for key features. Always visualize before running statistical tests — you'll catch patterns that numbers alone might miss.

In [ ]:
key_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'AMT_INCOME_TOTAL',
                'AMT_CREDIT', 'AGE_YEARS', 'CREDIT_INCOME_RATIO']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Reference (Training) vs Production Data Distribution', fontsize=14, fontweight='bold')

for idx, feature in enumerate(key_features):
    ax = axes[idx // 3][idx % 3]
    ax.hist(reference_data[feature].dropna(), bins=40, alpha=0.5,
            label='Reference (Training)', color='steelblue', density=True)
    ax.hist(current_data[feature].dropna(), bins=40, alpha=0.5,
            label='Production', color='darkorange', density=True)
    ax.set_title(feature, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visual inspection complete. Orange shifts indicate drift.")

## Section 4: Statistical Drift Testing with Evidently AI

Run formal statistical tests (Kolmogorov-Smirnov for numeric, Chi-square for categorical) to detect drift on every feature.

In [ ]:
from evidently.report import Report
from evidently.metrics import DatasetDriftMetric, DataDriftTable
import os

# Create drift report
drift_report = Report(metrics=[
    DatasetDriftMetric(),  # Overall: is there significant drift?
    DataDriftTable(),      # Per-feature: which features drifted?
])

drift_report.run(
    reference_data=reference_data,
    current_data=current_data,
)

# Save as interactive HTML for exploration
os.makedirs('monitoring', exist_ok=True)
drift_report.save_html("monitoring/drift_report.html")

print("✓ Drift report generated: monitoring/drift_report.html")
print("✓ Open in browser to view interactive results")

## Section 5: Extract and Display Results

Parse drift detection results programmatically for automated monitoring.

# Extract overall drift result
report_dict = drift_report.as_dict()
ds = report_dict['metrics'][0]['result']

print("="*70)
print("OVERALL DRIFT DETECTION")
print("="*70)
print(f"Dataset Drift Detected: {'YES ⚠️' if ds['dataset_drift'] else 'NO ✓'}")
print(f"Drifted Features: {ds['number_of_drifted_columns']} / {ds['number_of_columns']}")
print()

In [ ]:
# Per-feature drift details
drift_table = report_dict['metrics'][1]['result']

print("="*70)
print("PER-FEATURE DRIFT ANALYSIS")
print("="*70)
print(f"{'Feature':<25} {'Drifted?':<10} {'Score':<12} {'Test'}")
print("-" * 70)

for col, info in drift_table['drift_by_columns'].items():
    status = "YES ⚠️" if info['drift_detected'] else "no ✓"
    score = info['drift_score']
    test = info['stattest_name']
    print(f"{col:<25} {status:<10} {score:<12.6f} {test}")

print()

In [ ]:
## Section 6: Interpretation and Action Plan

What drift detection means and what to do about it.

In [ ]:
print("="*70)
print("INTERPRETATION")
print("="*70)

if ds['dataset_drift']:
    print("""
✓ DRIFT DETECTED — Action Required

What this means for the Home Credit model:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The data arriving in production is statistically different from
the data the model was trained on. THIS IS NORMAL for real systems.

Root causes (as simulated):
  1. EXT_SOURCE scores shifted (+0.05 on avg) → Credit bureaus recalibrated
  2. Financial amounts increased → Inflation/economic growth
  3. Applicant age decreased → New younger customer segment
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Recommended actions (priority order):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. IMMEDIATE: Evaluate model AUC on recent labeled production data.
   If AUC dropped below 0.70, the model is degraded.
   
2. SHORT-TERM: Investigate each drifted feature individually.
   Is this a data pipeline bug or a genuine real-world shift?
   
3. MEDIUM-TERM: If performance degraded, retrain the model
   using recent data that includes the new distributions.
   
4. LONG-TERM: Set up automated weekly drift monitoring
   with alerts when drift exceeds thresholds.
    """)
else:
    print("✓ No significant drift detected. Model is still relevant.")

print()

## Statistical Summary: Mean Shifts Across Features

Quantify how much each feature changed on average.

In [3]:
print("="*70)
print("QUANTITATIVE COMPARISON: FEATURE SHIFTS")
print("="*70)

# Align columns properly
ref_cols = reference_data.columns.tolist()
prod_cols = current_data[ref_cols].columns.tolist()

comp = pd.DataFrame({
    'Feature': ref_cols,
    'Train Mean': reference_data[ref_cols].mean().round(3).values,
    'Prod Mean': current_data[ref_cols].mean().round(3).values,
})

comp['Absolute Shift'] = (comp['Prod Mean'] - comp['Train Mean']).round(3)
comp['% Shift'] = ((comp['Prod Mean'] - comp['Train Mean']) / (comp['Train Mean'].abs() + 0.001) * 100).round(1)

print(comp.to_string(index=False))
print()
print("Interpretation:")
print("  • Positive shift = feature increased in production")
print("  • Negative shift = feature decreased in production")
print("  • Large % shifts (>10%) indicate significant drift")
print()

QUANTITATIVE COMPARISON: FEATURE SHIFTS


NameError: name 'reference_data' is not defined

In [ ]:
print("="*70)
print("MONITORING SETUP")
print("="*70)
print("""
To set up automated monitoring in production:

1. Collect predictions and inputs to data/reference_data.csv periodically
2. Run this notebook weekly to detect drift early
3. Set up alerts:
   - If dataset_drift = True: review model AUC
   - If any feature drifts significantly: investigate root cause
   - If AUC drops >5%: schedule retraining

Example cron job (weekly drift check):
  0 0 * * 0 cd /app && python -c "exec(open('notebooks/data_drift_analysis.ipynb').read())"

This ensures you're notified BEFORE the model silently degrades.
""")

In [ ]:
current_data = pd.DataFrame({
 'EXT_SOURCE_1': prod_ext_1,
 'EXT_SOURCE_2': prod_ext_2,
 'EXT_SOURCE_3': prod_ext_3,
 'AMT_INCOME_TOTAL': prod_income,
 'AMT_CREDIT': prod_credit,
 'AMT_ANNUITY': prod_annuity,
 'AMT_GOODS_PRICE': prod_goods,
 'AGE_YEARS': prod_age,
 'CODE_GENDER': reference_data['CODE_GENDER'].sample(n_prod, replace=True).values,
 'FLAG_OWN_CAR': reference_data['FLAG_OWN_CAR'].sample(n_prod, replace=True).values,
 'FLAG_OWN_REALTY': reference_data['FLAG_OWN_REALTY'].sample(n_prod, replace=True).values,
 'CNT_CHILDREN': reference_data['CNT_CHILDREN'].sample(n_prod, replace=True).values,
 'YEARS_EMPLOYED': reference_data['YEARS_EMPLOYED'].sample(n_prod, replace=True).values,
 'YEARS_ID_PUBLISH': reference_data['YEARS_ID_PUBLISH'].sample(n_prod, replace=True).values,
 'EDUCATION_LEVEL': reference_data['EDUCATION_LEVEL'].sample(n_prod, replace=True).values,
})

current_data['CREDIT_INCOME_RATIO'] = current_data['AMT_CREDIT'] / (current_data['AMT_INCOME_TOTAL'] + 1)
current_data['ANNUITY_INCOME_RATIO'] = current_data['AMT_ANNUITY'] / (current_data['AMT_INCOME_TOTAL'] + 1)
current_data['CREDIT_GOODS_RATIO'] = current_data['AMT_CREDIT'] / (current_data['AMT_GOODS_PRICE'] + 1)

In [ ]:
import matplotlib.pyplot as plt

key_features = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'AMT_INCOME_TOTAL',
 'AMT_CREDIT', 'AGE_YEARS', 'CREDIT_INCOME_RATIO']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Reference (blue) vs Production (orange)', fontsize=14)

for idx, feature in enumerate(key_features):
 ax = axes[idx // 3][idx % 3]
 ax.hist(reference_data[feature].dropna(), bins=40, alpha=0.5,
 label='Reference', color='steelblue', density=True)
 ax.hist(current_data[feature].dropna(), bins=40, alpha=0.5,
 label='Production', color='darkorange', density=True)
 ax.set_title(feature)
 ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 10.4 — Run Evidently drift report
Using Evidently AI to run formal statistical tests for drift on every feature. Visual comparison is good for getting intuition, but you need statistical rigor for automated decisions. Evidently uses appropriate statistical tests (Kolmogorov-Smirnov for numeric features, chi-square for categorical) and reports whether each feature has significantly drifted. (Evidently AI docs)

In [ ]:
from evidently.report import Report
from evidently.metrics import DatasetDriftMetric, DataDriftTable

drift_report = Report(metrics=[
 DatasetDriftMetric(), # Overall: is there significant drift?
 DataDriftTable(), # Per-feature: which features drifted?
])

drift_report.run(
 reference_data=reference_data,
 current_data=current_data,
)

# Save as interactive HTML
import os
os.makedirs('monitoring', exist_ok=True)
drift_report.save_html("monitoring/drift_report.html")

## 10.5 — Extract and display results
Extracting the drift results programmatically. The HTML report is great for humans exploring interactively. But for automated monitoring (e.g., a weekly script that sends an alert if drift is detected), you need to extract results as data.

In [ ]:
report_dict = drift_report.as_dict()

ds = report_dict['metrics'][0]['result']
print(f"Overall drift detected: {'YES' if ds['dataset_drift'] else 'NO'}")
print(f"Drifted features: {ds['number_of_drifted_columns']} / {ds['number_of_columns']}")

In [ ]:
drift_table = report_dict['metrics'][1]['result']

print(f"\n{'Feature':<25} {'Drifted?':<10} {'Score':<12} {'Test'}")
print("-" * 65)

for col, info in drift_table['drift_by_columns'].items():
 status = "YES" if info['drift_detected'] else "no"
 score = info['drift_score']
 test = info['stattest_name']
 flag = " << ALERT" if info['drift_detected'] else ""
 print(f"{col:<25} {status:<10} {score:<12.6f} {test}{flag}")

## 10.6 — Interpretation and action plan
Translating statistical results into business actions. Detecting drift is the easy part. The hard part — and the part that actually matters — is deciding what to do about it. A drift detection without an action plan is just an interesting observation.

In [ ]:
if ds['dataset_drift']:
 print("""
 DRIFT DETECTED — Action Required

 What this means for the Home Credit model:
 The data arriving in production is statistically different from
 the data the model was trained on. Specific shifts identified:
 - EXT_SOURCE scores shifted (credit bureau scoring updates)
 - Financial amounts increased (inflation / economic growth)
 - Applicant age decreased (new younger customer segment)
 
 Recommended actions (in priority order):
 1. IMMEDIATE: Evaluate model AUC on recent labeled production data.
 If AUC dropped below 0.70, the model is degraded.
 2. SHORT-TERM: Investigate each drifted feature individually.
 Is this a data pipeline bug or a genuine real-world shift?
 3. MEDIUM-TERM: If performance degraded, retrain the model
 using recent data that includes the new distributions.
 4. LONG-TERM: Set up automated weekly drift monitoring
 with alerts when drift exceeds thresholds.
 """)

Creating a statistical comparison table. This table gives you concrete numbers to share with stakeholders. "AMT_CREDIT increased 12% on average" is more actionable than "drift was detected on AMT_CREDIT."

In [ ]:
comp = pd.DataFrame({
 'Feature': reference_data.columns,
 'Train Mean': reference_data.mean().round(2).values,
 'Prod Mean': current_data[reference_data.columns].mean().round(2).values,
})
comp['Shift %'] = ((comp['Prod Mean'] - comp['Train Mean']) / (comp['Train Mean'] + 0.001) * 100).round(1)
print(comp.to_string(index=False))